Ich begann meinen Task damit, erstmal zu schauen, ob mein Task der Vorhersage "Wann ein Kommentar am wahrscheinlichsten eintritt" überhaupt möglich ist. Dafür habe ich erstmal geguckt, ob eine Klassifizierung möglich ist. Dabei habe ich mir überlegt die Inputdaten in das Modell zu geben und das Modell, basierend auf den Trainingsdaten, als Output für jeden Satz zu sagen, ob es einen Einwurf enthält (mit 1 gelabelt) oder nicht (mit 0 gelabelt).

Im folgenden Notebook kann man eine Textklassifizierung mit Hilfe des Naive-Bayes sehen. Gründe warum ich mich für den Naive-Bayes entschieden habe:
1. Einfachheit & Effizienz: Der Algorithmuss kann relativ schnell über einen großen Korpus laufen.
2. Handhabung von Sparse Data: Kann dünn besetze Matrizen gut verarbeiten. Mit dünn besetzt ist hier vor allem gemeint, dass die meisten Daten in der Merkmalsmatrix null sind. Verhältnismäßig sind nur wenige 1. 

Es gibt mehrere Arten des Naive Bayes. Welchen man wählt, hängt von den Daten ab. Ich habe mich explizit für den MultinomialNB entschieden, da z.B. der Gaussian Naive Bayes annimmt, dass die Daten normalverteilt sind (was hier definitiv nicht der Fall ist). Man könnte denken, dass sich der BernoulliNB anbieten könnte. Dieser klassifiziert, ob ein bestimmtes Wort im Datensatz vorkommt oder nicht. Er beachtet jedoch nicht die Häufigkeit von Wörtern. Der MultinomialNB tut dies aber schon und ist deshalb in der Lage Muster in den Daten zu erkennen, die durch die Häufigkeit bestimmter Wörter gekennzeichnet sind. Damit ist das Modell in der Lage auf Inputdaten ohne "interruption" Token sinnvolle Vorhersagen zu treffen.

Ich habe auf mehrere, nützliche Funktionen aus der sklearn-Library zurückgegriffen. 
1. CountVectorizer wandelt die Textdaten in eine Matrix um, die als Eingabe für das Modell dient.
2. MultinomialNB ist der Naive-Bayes Klassifikator für die Modellierung, den wir später trainieren werden.
3. train_test_split ist sehr nützlich, um die Daten in Trainings- und Testdatensätze zu unterteilen.
4. accuracy score berechnet die Genauigkeit der Vorhersagen des Modells

Die folgende Zelle diente nur dazu mich mit der Implementierung des Naive Bayes vertraut zu machen. Der Korpus ist also nicht sehr qualitativ hochwertig, da ich Beispielsätze geschrieben und in einer Liste gespeichert habe und diese am Ende mit einer beliebigen Zahl (hier 7) multipliziert habe, um zu sehen, wie sich der Naive Bayes mit der Anzahl der Daten verändert.

Naive-Bayes mit 700 Trainingsdaten:

In [48]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import random

#700 Beispiel-Sätze 
text = [
    "Das ist ein Beispieltext.", "Hier könnte ein Kommentar sein.", "Ein weiterer Satz ohne Kommentar.",
    "Und noch ein Satz mit einem möglichen Kommentar.", "Dieser Satz hat keinen Kommentar.",
    "Vielleicht hier ein Kommentar?", "Ein Satz ohne besondere Bedeutung.", "Könnte hier ein Kommentar sein?",
    "Wieder ein normaler Satz.", "Ein Kommentar wäre hier denkbar.", "Das ist ein Test.",
    "Kein Kommentar in diesem Satz.", "Ein weiterer Beispielsatz.", "Hier könnte ein Kommentar stehen.",
    "Normaler Satz ohne Kommentar.", "Eventuell ein Kommentar hier.", "Ein Satz ohne Kommentar.",
    "Möglicher Kommentar hier.", "Beispielsatz ohne Kommentar.", "Könnte das ein Kommentar sein?",
    "Wieder ein Satz ohne Kommentar.", "Hier eventuell ein Kommentar.", "Kein Kommentar in diesem Satz.",
    "Noch ein normaler Satz.", "Vielleicht ein Kommentar hier?", "Das ist wieder ein Beispielsatz.",
    "Kommentar möglich.", "Ein Satz ohne Kommentar.", "Möglicher Kommentar hier.",
    "Normaler Satz ohne besondere Bedeutung.", "Vielleicht ein Kommentar hier?", "Ein Satz ohne besondere Merkmale.",
    "Könnte hier ein Kommentar stehen?", "Wieder ein normaler Satz.", "Ein Kommentar wäre hier denkbar.",
    "Das ist ein weiterer Test.", "Kein Kommentar in diesem Satz.", "Ein weiterer Beispielsatz.",
    "Hier könnte ein Kommentar stehen.", "Normaler Satz ohne Kommentar.", "Eventuell ein Kommentar hier.",
    "Ein Satz ohne Kommentar.", "Möglicher Kommentar hier.", "Beispielsatz ohne Kommentar.",
    "Könnte das ein Kommentar sein?", "Wieder ein Satz ohne Kommentar.", "Hier eventuell ein Kommentar.",
    "Kein Kommentar in diesem Satz.", "Noch ein normaler Satz.", "Vielleicht ein Kommentar hier?",
    "Das ist ein Beispieltext.", "Hier könnte ein Kommentar sein.", "Ein weiterer Satz ohne Kommentar.",
    "Und noch ein Satz mit einem möglichen Kommentar.", "Dieser Satz hat keinen Kommentar.",
    "Vielleicht hier ein Kommentar?", "Ein Satz ohne besondere Bedeutung.", "Könnte hier ein Kommentar sein?",
    "Wieder ein normaler Satz.", "Ein Kommentar wäre hier denkbar.", "Das ist ein Test.",
    "Kein Kommentar in diesem Satz.", "Ein weiterer Beispielsatz.", "Hier könnte ein Kommentar stehen.",
    "Normaler Satz ohne Kommentar.", "Eventuell ein Kommentar hier.", "Ein Satz ohne Kommentar.",
    "Möglicher Kommentar hier.", "Beispielsatz ohne Kommentar.", "Könnte das ein Kommentar sein?",
    "Wieder ein Satz ohne Kommentar.", "Hier eventuell ein Kommentar.", "Kein Kommentar in diesem Satz.",
    "Noch ein normaler Satz.", "Vielleicht ein Kommentar hier?", "Das ist wieder ein Beispielsatz.",
    "Kommentar möglich.", "Ein Satz ohne Kommentar.", "Möglicher Kommentar hier.",
    "Normaler Satz ohne besondere Bedeutung.", "Vielleicht ein Kommentar hier?", "Ein Satz ohne besondere Merkmale.",
    "Könnte hier ein Kommentar stehen?", "Wieder ein normaler Satz.", "Ein Kommentar wäre hier denkbar.",
    "Das ist ein weiterer Test.", "Kein Kommentar in diesem Satz.", "Ein weiterer Beispielsatz.",
    "Hier könnte ein Kommentar stehen.", "Normaler Satz ohne Kommentar.", "Eventuell ein Kommentar hier.",
    "Ein Satz ohne Kommentar.", "Möglicher Kommentar hier.", "Beispielsatz ohne Kommentar.",
    "Könnte das ein Kommentar sein?", "Wieder ein Satz ohne Kommentar.", "Hier eventuell ein Kommentar.",
    "Kein Kommentar in diesem Satz.", "Noch ein normaler Satz.","Noch ein normaler Satz."
] * 7  # Die Sätze werden mit 7 multipliziert, um insgesamt 700 Sätze zu erhalten

# Zufällige Auswahl von 70 Sätzen des Korpus
comment_indices = random.sample(range(700), 70)

# Labelgenerierung
labels = [1 if i in comment_indices else 0 for i in range(700)]

# Überprüfen, ob die Länge der Texte und Labels gleich ist (war Teil des Debugging)
assert len(text) == len(labels), f"Länge von Texten ({len(text)}) und Labels ({len(labels)}) stimmt nicht überein."

data = {
    'text': text, 
    'label': labels
}

df = pd.DataFrame(data)

# Text wird in Merkmale umgewandelt
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['text'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Genauigkeit: {accuracy * 100:.2f}%")


Genauigkeit: 85.71%


Folgend habe ich das Modell auf meine taskspezifischen Daten der Bundestagsreden angepasst. Dafür habe ich erstmal versucht das Modell auf bekannten Daten richtig predicten zu lassen. Mit bekannten Daten war also gemeint, dass in den Inputdaten "interruption" Token vorhanden waren. Dies erklärt auch die hohe Accuracy. Es ist für das Modell viel leichter die "interruption" Token zu erkennen, als die Muster zu erkennen, wo diese Token eigentlich stehen würden, weil das ja kein wirkliches Lernen von Mustern voraussetzt.

Was bei der Evaluation auffällt, ist, dass precision und recall bei der Klasse 1 (Einwurf) bei nahezu 0 sind.

Begründung: Das Modell sieht während des Trainings viel mehr Beispiele der Klasse 0 als der Klasse 1. Daher lernt es häufiger die Klasse 0 vorherzusagen und die Modellleistung für die Klasse 1 wird beeinträchtigt. Hier tritt das gleiche Problem, wie beim [nicht gefinetuned BERT Modell](./IV_interruption_prediction/Every_Model_As_A_Different_Approach_And_Evaluation/02_Unfinetuned_BERT.ipynb) auf. 
Betrachten wir nun die Precision mit einem Wert von 0,02. Daraus lässt sich folgern, dass von allen als 1 ("Einwurf") vorhergesagten Instanzen nur 2% tatsächlich Einwurfe sind (auch wenn das in den ersten 21 Zeilen nicht so aussieht).
Bei Betrachtung des Recalls lässt sich folgern, dass fast keine tatsächlichen Einwürfe korrekt identifiziert wurden.

In [6]:
import json
import re
from sklearn.metrics import accuracy_score, classification_report
from IPython.display import display

file_path = '/var/lib/private/simonjaned87761/team-04_deprecated/data/opendata_api/modified_texts_with_interruption.json'

with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# Funktion zum Aufteilen der Daten in Sätze
def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?]) +', text)
    return sentences

#Generierung der Labels aus den ersten 1000 Einträgen: 1 bei <interruption>, sonst 0
texts = []
labels = []
for entry in data[:1000]:  
    sentences = split_into_sentences(entry['modified_text'])
    for sentence in sentences:
        texts.append(sentence)
        labels.append(1 if '<interruption>' in sentence else 0)

df = pd.DataFrame({'text': texts, 'label': labels})

# Entfernen der <interruption> Token aus den Texten, um das Modell auch Muster leichter lernen zu lassen
# dient zur Vorhersage von Einwürfen auf Daten, wo nicht explizit <interruption> steht
df['text'] = df['text'].str.replace('<interruption>', '', regex=False)

vectorizer = CountVectorizer(binary=True)  # Binary zählt nur die Anwesenheit von Wörtern
X = vectorizer.fit_transform(df['text'])
y = df['label']

# Train-Test-Split mit stratified sampling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Genauigkeit: {accuracy * 100:.2f}%")

print("Klassifikationsbericht:\n", classification_report(y_test, y_pred))

# Funktion zur Extraktion von Sätzen und Labels aus einem bestimmten Eintrag
def extract_sentences_from_entry(entry_index):
    entry = data[entry_index]
    sentences = split_into_sentences(entry['modified_text'])
    entry_texts = []
    entry_labels = []
    for sentence in sentences:
        entry_texts.append(sentence)
        entry_labels.append(1 if '<interruption>' in sentence else 0)
    return entry_texts, entry_labels

# Festlegen des Eintrags, aus dem die ersten 50 Sätze genommen werden
entry_index = 6  # Beispiel: Eintrag 6 (diente zum Ausprobieren, da jeder Eintrag verschieden viele <interruption> enthält)
entry_texts, entry_labels = extract_sentences_from_entry(entry_index)

entry_df = pd.DataFrame({'text': entry_texts, 'label': entry_labels})
entry_df['text'] = entry_df['text'].str.replace('<interruption>', '', regex=False)

# Verteilung der Labels ausgeben
print("Verteilung der Labels:\n", df['label'].value_counts())

# Ausgabe der ersten 20 Sätze zusammen mit den Vorhersagen
entry_df['Predicted Label'] = model.predict(vectorizer.transform(entry_df['text']))
display(entry_df[['text', 'label', 'Predicted Label']].head(21))

Genauigkeit: 92.50%
Klassifikationsbericht:
               precision    recall  f1-score   support

           0       0.93      0.99      0.96      5831
           1       0.02      0.00      0.00       411

    accuracy                           0.93      6242
   macro avg       0.48      0.50      0.48      6242
weighted avg       0.87      0.93      0.90      6242

Verteilung der Labels:
 label
0    29156
1     2053
Name: count, dtype: int64


,text,label,Predicted Label
0,Herr Präsident!,0,0
1,Meine sehr verehrten Damen und Herren!,0,0
2,Nun hat sich die Große Koalition 50 Minuten la...,1,0
3,Deshalb: Lassen Sie uns mehr Realität wagen!,0,0
4,Gerade noch 15 Millionen Nettosteuerzahler fin...,0,0
5,Das ist fast die gesamte Bevölkerung Dänemarks...,0,0
6,Vor fünf Jahren waren es fast 600 000 weniger....,0,0
7,Birkwald [DIE LINKE].,0,0
8,Noch keine 30 Sekunden hat er diesmal gebrauch...,0,0
9,Dafür haben Sie schon Schwielen an den Händen ...,1,1


Folgend habe ich es nun auf unvorhergesehenen Daten ohne "interruption" Token getestet. Man kann bei der Ausgabe erkennen, dass die Vorhersage relativ akkurat ist, obwohl die Verteilung der Labels sehr unausgeglichen ist. Labels mit 0 waren zu 93,4% vertreten und Labels mit 1 nur zu 6,6%. Auf diesen Inputdaten, konnte das Modell 2 von 3 korrekte Vorhersagen machen.

Begründung: Die Menge der Inputdaten (new_data_texts) ist sehr klein und die Verteilung der Klassen ist weniger unausgewogen als der ursprüngliche Datensatz. Die bessere Performance wird dadurch begründet, dass es weniger durch die extreme Klassenverteilung, wie im ursprünglichen Datensatz beeinflusst wird.

In [7]:
# Funktion, um den MultinomialNB auf neue Daten Vorhersagen treffen zu lassen
def predict_on_new_data(new_texts, true_labels):
    new_df = pd.DataFrame({'text': new_texts})
    new_X = vectorizer.transform(new_df['text'])
    new_df['Predicted Label'] = model.predict(new_X)
    new_df['True Label'] = true_labels
    
    print("Verteilung der Labels:\n", df['label'].value_counts())

    accuracy = accuracy_score(new_df['True Label'], new_df['Predicted Label'])
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("Klassifikationsbericht:\n", classification_report(new_df['True Label'], new_df['Predicted Label']))
    
    return new_df

# Beispieldaten ohne <interruption> Token
new_data_texts = [
    "Herr Präsident! Liebe Kolleginnen! Liebe Kollegen! In der Krise zeigt sich der Charakter. Das gilt für den Einzelnen, zum Guten wie zum Schlechten; das kennen wir. Das gilt auch für einzelne Fraktionen, wenn ich an das spalterisch-hetzerische Reden auf der äußeren Rechten denke.",
    "Wer bei staatlicher Unterstützung zunächst nach dem Abstammungsnachweis fragt – der völkische Versorger ist das Gegenteil von einem menschengerechten Sozialstaat –, der hat von Menschenwürde und Sozialstaatlichkeit nichts verstanden.",
    "Krise zeigt Charakter – das gilt auch für unser Gemeinwesen, für unseren Staat als Ganzes. Darin zeigt sich: Unser Land ist geprägt durch eine solidarische Gesellschaft und einen starken Sozialstaat. Darauf sind wir stolz, meine Damen, meine Herren.",
    "Diese 2 Millionen nichtdeutschen Leistungsbezieher kosten uns etwa 13 Milliarden Euro im Jahr, und da sind die Ausgaben für Asylbewerber und ausländische Sozialhilfeempfänger noch nicht einmal mit eingerechnet.",
    "Das alles erwähnen Sie hier mit keinem einzigen Wort. Dafür haben Sie schon Schwielen an den Händen vom gegenseitigen Schulterklopfen für Ihre Grundrente, die Sie dreisterweise „Respektrente“ nennen, für die Sie aber gerade einmal 1,3 Milliarden Euro aufbringen, also nur ein Zehntel dessen, was Sie für ausländische Hartz-IV-Empfänger zahlen.",
    "Sie speisen unsere Eltern und Großeltern mit Armutsrenten ab, während Sie die Früchte ihrer Lebensleistung völlig verantwortungslos an alle Welt verschleudern.",
    "Von 2012 bis 2019, also in nur sieben Jahren, sind rund 4 Millionen Menschen aus aller Welt nach Deutschland gekommen. Das ist etwa die Einwohnerzahl von Brandenburg und Mecklenburg-Vorpommern zusammengenommen. Im Gegenzug haben 3,4 Millionen Deutsche, etwa die Einwohnerzahl Berlins, das Land verlassen: meist hochqualifizierte Fachkräfte, die woanders bessere Perspektiven vorfinden als im ausgemerkelten Land mit der höchsten Steuer- und Abgabenlast der Welt.",
    "Die Leistungsträger von heute fliehen, und die, die unseren Wohlstand aufgebaut haben, verarmen. Seit Amtsantritt der Bundeskanzlerin hat sich die Zahl der Rentner, die auf Sozialhilfe angewiesen sind, fast verdoppelt. 1,3 Millionen Altersrentner müssen zusätzlich arbeiten gehen. Da sind die Flaschensammler, die man leider immer häufiger in unseren Stadtbildern sieht, nicht mit dabei.",
    "Und dennoch wollen Sie im vorliegenden Haushalt, dass der Bund jetzt auch noch zu 100 Prozent die Kosten der Unterkunft für jene neuen Asylforderer übernimmt, die einzelne Kommunen noch zusätzlich aufnehmen wollen, darunter die etwa 220 Kommunen der Initiative Seebrücke, die auch von der Antifa und anderen gewaltbereiten Linken unterstützt wird,",
    "im Übrigen auch vom Potsdamer Oberbürgermeister, Ihrem SPD-Kollegen, Herr Minister.",
    "Das heißt, zumeist links-grün regierte Kommunen locken nun als sogenannte sichere Häfen eigenmächtig Asylsucher ins Land, während wir angesichts drohender Massenarbeitslosigkeit und einer Viertelmillion Ausreisepflichtiger in Deutschland vielmehr eine Abschiebeoffensive bräuchten.",
    "Der deutsche Arbeitnehmer darf diese Menschen dann über den Bundeshaushalt mittels Hartz IV und Wohnkosten voll finanzieren, während er selbst die höchste Lebensarbeitszeit aller Euro-Länder ableisten muss, die niedrigsten Renten bekommt und am wenigsten Vermögen in der Tasche hat.",
    "Nun wollen Sie im kommenden Jahr mehr als 164 Milliarden Euro für Sozialpolitik ausgeben. Das sind über 26 Milliarden Euro mehr als im letzten Jahr. Allein diese Steigerung von 26 Milliarden Euro ist mehr, als Sie insgesamt in Bildung und Forschung stecken, und doppelt so hoch wie das, was Sie für Familien ausgeben wollen. Das zeigt, wo Ihre Prioritäten liegen. Ihre Prioritäten liegen nicht da, wo wir sie als AfD sehen. Sie haben unsere Kinder zum Armutsrisiko gemacht und unsere Rentner zu Bedürftigen.",
    "Mit diesem Haushalt haben Sie schwarz auf weiß den Nachweis erbracht, dass Ihre Politik gegen die eigenen Bürger gerichtet ist. Und das ist exakt das Gegenteil der Politik, die wir als AfD vertreten.",
    "Die AfD-Fraktion lehnt diesen Haushalt ab."
]

# hier habe ich manuell die True Labels sinnvoll vergeben
new_data_labels = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]

new_predictions = predict_on_new_data(new_data_texts, new_data_labels)

display(new_predictions[['text', 'True Label', 'Predicted Label']])

Verteilung der Labels:
 label
0    29156
1     2053
Name: count, dtype: int64
Accuracy: 93.33%
Klassifikationsbericht:
               precision    recall  f1-score   support

           0       1.00      0.92      0.96        13
           1       0.67      1.00      0.80         2

    accuracy                           0.93        15
   macro avg       0.83      0.96      0.88        15
weighted avg       0.96      0.93      0.94        15



,text,True Label,Predicted Label
0,Herr Präsident! Liebe Kolleginnen! Liebe Kolle...,0,0
1,Wer bei staatlicher Unterstützung zunächst nac...,1,1
2,Krise zeigt Charakter – das gilt auch für unse...,0,0
3,Diese 2 Millionen nichtdeutschen Leistungsbezi...,0,0
4,Das alles erwähnen Sie hier mit keinem einzige...,0,0
5,Sie speisen unsere Eltern und Großeltern mit A...,0,1
6,"Von 2012 bis 2019, also in nur sieben Jahren, ...",0,0
7,"Die Leistungsträger von heute fliehen, und die...",0,0
8,Und dennoch wollen Sie im vorliegenden Haushal...,0,0
9,im Übrigen auch vom Potsdamer Oberbürgermeiste...,0,0
